<h1>DOCUMENT LOADER</h1>

In [63]:
from langchain_community.document_loaders import Docx2txtLoader
import os 

def document_loader(folder_path):

    document = []

    for filename in os.listdir(folder_path):

        if filename.endswith('.docx'):
            file_path = os.path.join(folder_path,filename)

            loader = Docx2txtLoader(file_path)
            pages = loader.load()


            document.extend(pages)
    return document

<h1>Text Splitter</H1>

In [64]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def Text_Splitter(document):

    Text_Splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 60
    )

    chunk = Text_Splitter.split_documents(document)

    return chunk



<h1>EMBEDDING AND VECTOR DATABASE</h1>

In [65]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

def create_vector_db(chunk):

    embedding = HuggingFaceEmbeddings(
        model = 'sentence-transformers/all-MiniLM-L6-v2'
    )

    vector_db = FAISS.from_documents(
        chunk,
        embedding
    )

    return vector_db

<h1>KEYWORD SEARCHING</h1>

In [66]:
from rank_bm25 import BM25Okapi

class keyword_search:

    def __init__(self,chunk):
        self.chunk = chunk
        tokenized_chunk = []

        for t in chunk:
            token = t.page_content.lower().split()
            tokenized_chunk.append(token)


        self.bm250 = BM25Okapi(tokenized_chunk)

    def search(self,question,k=5):

            query = question.lower().split()

            scores = self.bm250.get_scores(query)

            ranked_answers = sorted(range(len(scores)),
                                    key = lambda index:scores[index],
                                    reverse=True)

            top_answers = ranked_answers[:k]

            result = []

            for r in top_answers:
                 result.append(self.chunk[r])
        
            return result

<h1>HYBRID SEARCH</h1>

In [67]:
class Hybrid_Search:

    def __init__(self,vector,bm250):
        self.vector = vector
        self.bm250 = bm250

    def combine_search(self,query,k=5):

        vector_search = self.vector.similarity_search(query,k=k)
        bm250_search = self.bm250.search(query,k=k)

        combine_search_hybrid = vector_search+bm250_search


        answers = []
        unseen_content = set()
        for x in combine_search_hybrid:
            if x.page_content not in combine_search_hybrid:
                answers.append(x)
                unseen_content.add(x.page_content)
        return answers


<h1>RERANKER</h1>

In [68]:
from sentence_transformers import CrossEncoder

class RERANKER:

    def __init__(self):
        self.model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    def rerank(self,query,document,top_k=5):

        pairs = []

        for c in document:

            pairs.append(
                [
                    query,
                    c.page_content
                ]
            )
        scores = self.model.predict(pairs)

        ranked_documents = sorted(zip(scores,document),
                                key = lambda i: i[0],
                                reverse=True)

        top_answers = ranked_documents[:top_k]

        result = []

        for scores,document in top_answers:
            result.append(document)
        return result

<h1>PROMPT</h1>

In [73]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from dotenv import load_dotenv


def generate_content(document,query):
    context = ""
    for c in document:
        context += c.page_content
        context+= "\n\n"

    prompt = ChatPromptTemplate.from_template(
        '''
        You have to answer question from the given context.

        if the question is out of context. just say,"Question out of context"

        context
        {context}

        question
        {question}
        
'''
    )

    llm_model = ChatGroq(
        model = 'openai/gpt-oss-20b',
        temperature=0
    )


    chain = prompt | llm_model

    response = chain.invoke({
        'context':context,
        'question':query
    })

    return response.content


<h1>APP</h1>

In [74]:
import gradio as gr


folder_path = 'E:\GEN-AI-PROJECTS'

# document loader
document = document_loader(folder_path)

# text splitter
chunks = Text_Splitter(document)

# create vector db
vectorDB = create_vector_db(chunks)

#create keyword search
bm250 = keyword_search(chunks)

# hybrid search

hybridSearch = Hybrid_Search(vectorDB,bm250)


#reranker
reranker = RERANKER()

def genrate_answer(query):

    retirved_document = hybridSearch.combine_search(query,k=5)

    rereanked_final = reranker.rerank(query,retirved_document,top_k=5)

    answer = generate_content(rereanked_final,query)


    sources = ""

    for i,document in enumerate(rereanked_final):

        page = document.metadata.get(
            'page',
            'unknown'
        )

        sources += f"\nSource {i+1} page {page}"

    final_response = answer
    final_response += '\n\nSource:'
    final_response += sources

    return final_response


demo = gr.Interface(
    fn = genrate_answer,
    inputs = gr.Textbox(
        label = 'Ask a question'
    ),
    outputs=gr.Textbox(
        label = 'Answer',
        lines = 15
    ),
    
        title = 'DEEP KNOWLWDGE',
        description='Advanced RAG Knowledge Base'
)

demo.launch(share=True)


    





Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

* Running on local URL:  http://127.0.0.1:7867

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
